# SkinLesionNet — HAM10000 training, feature analysis and app export

Run in Google Colab. Downloads the published HAM10000 package to temporary Colab storage; no Google Drive mount or uploaded files. The package is about 3.2 GB; extracted data uses additional temporary disk space. It reuses a complete extraction and resumes partial downloads in the same runtime.

HAM10000 is **non-commercial use only**. Read its terms before running: https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/DBW86T. Starting the download is your choice to proceed under those terms; there is no True/False gate.

This workflow inspects metadata (age, sex, body location, lesion ID and diagnosis-confirmation type), checks exact duplicate images, creates lesion/duplicate-grouped train/validation/test splits, and compares image-only against metadata-fusion EfficientNet-B0. The model may use age, sex and location; missing values are supported. IDs and diagnosis-confirmation type are analysis-only and never model inputs. Validation selects model and calibration temperature; the test set is kept for final evaluation.

It exports test metrics, per-class results, confusion matrix, calibration and subgroup summaries, training plots, an app-compatible checkpoint, and a downloadable ZIP. This is internal research evaluation, not clinical validation. HAM10000 has no patient IDs, so lesion grouping does not guarantee patient-level separation.

In [ ]:
%pip -q install timm scikit-learn pandas pillow matplotlib requests tqdm
import os, json, random, hashlib, zipfile, time
from pathlib import Path
import requests, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, torch.nn as nn, torch.nn.functional as F, timm
from tqdm.auto import tqdm
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report, log_loss
from IPython.display import display

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if DEVICE.type=="cuda" else "none")
if DEVICE.type!="cuda": print("CPU training is slow. Choose Runtime > Change runtime type > GPU before training.")
CLASSES=["akiec","bcc","bkl","df","mel","nv","vasc"]; C2I={c:i for i,c in enumerate(CLASSES)}
ROOT=Path("/content/ham10000"); OUT=Path("/content/skinlesion_run")
ROOT.mkdir(parents=True,exist_ok=True); OUT.mkdir(parents=True,exist_ok=True)
IMG_SIZE=224; BS=32 if DEVICE.type=="cuda" else 8; EPOCHS=18; PATIENCE=5; WORKERS=2

## 1. Download, resume and safely extract HAM10000

The published Dataverse package is downloaded once with a progress bar. If extraction is already complete, the download is skipped. Partial downloads are resumed where the host supports byte ranges. The outer archive and nested image archives are integrity-checked and extracted with path traversal protection. The metadata is discovered by name whether or not it has a .csv extension.

In [ ]:
URL="https://dataverse.harvard.edu/api/access/dataset/:persistentId/"
PARAMS={"persistentId":"doi:10.7910/DVN/DBW86T"}
ARCHIVE=Path("/content/HAM10000_dataverse.zip"); PART=Path("/content/HAM10000_dataverse.zip.partial")
def meta_files():
    return [p for p in ROOT.rglob("*") if p.is_file() and p.name.casefold().startswith("ham10000_metadata") and p.suffix.casefold() in ("",".csv",".txt")]
def jpg_files():
    return [p for p in ROOT.rglob("*") if p.is_file() and p.suffix.casefold() in (".jpg",".jpeg")]
def safe_extract(path,dest):
    dest=Path(dest).resolve()
    with zipfile.ZipFile(path) as z:
        bad=z.testzip()
        if bad: raise RuntimeError("Corrupt ZIP member: "+bad)
        for m in z.infolist():
            target=(dest/m.filename).resolve()
            if target!=dest and dest not in target.parents: raise RuntimeError("Unsafe archive path: "+m.filename)
        z.extractall(dest)
def download():
    offset=PART.stat().st_size if PART.exists() else 0
    headers={"Range":"bytes="+str(offset)+"-"} if offset else {}
    with requests.get(URL,params=PARAMS,headers=headers,stream=True,timeout=(30,180)) as r:
        r.raise_for_status(); append=offset>0 and r.status_code==206
        if offset and not append: offset=0
        total=int(r.headers.get("Content-Length",0))+offset
        with PART.open("ab" if append else "wb") as f, tqdm(total=total or None,initial=offset,unit="B",unit_scale=True,desc="HAM10000 download",mininterval=2) as bar:
            for chunk in r.iter_content(8*1024*1024):
                if chunk: f.write(chunk); bar.update(len(chunk))
    PART.replace(ARCHIVE)
    print("Package downloaded:",round(ARCHIVE.stat().st_size/1e9,2),"GB")
if not meta_files() or len(jpg_files())<10000:
    if not ARCHIVE.exists():
        print("The published package is approximately 3.2 GB and will use Colab temporary storage.")
        download()
    print("Validating and extracting package...")
    safe_extract(ARCHIVE,ROOT); ARCHIVE.unlink(missing_ok=True)
for round_no in range(1,4):
    nested=[p for p in ROOT.rglob("*.zip") if p.is_file()]
    if not nested: break
    print("Extracting",len(nested),"nested archive(s), round",round_no)
    for p in nested:
        target=p.with_suffix(""); target.mkdir(parents=True,exist_ok=True)
        safe_extract(p,target); p.unlink()
MFILES=meta_files(); IMAGES=jpg_files()
if len(MFILES)!=1: raise RuntimeError("Expected one HAM10000_metadata file; found "+repr([str(p) for p in MFILES]))
if len(IMAGES)<10000: raise RuntimeError("Found metadata but only "+str(len(IMAGES))+" JPEGs. Rerun in the same Colab runtime; partial file: "+str(PART))
META_PATH=MFILES[0]
print("Ready:",len(IMAGES),"JPEGs; metadata:",META_PATH.name,". Data stays in Colab runtime.")

## 2. Validate and explore dataset features

Model inputs are age, sex and localization. lesion_id and image_id are used only for grouping/lookups. dx_type is reserved for descriptive analysis: it describes how the diagnosis was confirmed and could leak label-adjacent information. Age is imputed/scaled from training data only; sex/localization are one-hot encoded with unknown categories. All feature preprocessing is exported for app inference.

In [ ]:
raw=pd.read_csv(META_PATH); raw.columns=[str(c).strip().lower() for c in raw.columns]
required={"image_id","lesion_id","dx","age","sex","localization","dx_type"}
if required-set(raw.columns): raise RuntimeError("Missing metadata columns: "+repr(sorted(required-set(raw.columns))))
raw["image_id"]=raw.image_id.astype(str).str.strip(); raw["lesion_id"]=raw.lesion_id.astype(str).str.strip()
raw["label"]=raw.dx.astype(str).str.lower().str.strip(); raw["age"]=pd.to_numeric(raw.age,errors="coerce")
for c in ["sex","localization","dx_type"]: raw[c]=raw[c].fillna("unknown").astype(str).str.strip().str.lower().replace({"":"unknown","nan":"unknown"})
if raw.image_id.duplicated().any() or set(raw.label)!=set(CLASSES): raise RuntimeError("Duplicate IDs or unexpected diagnosis labels.")
lookup={p.stem:p for p in IMAGES}; raw["image_path"]=raw.image_id.map(lookup)
if raw.image_path.isna().any(): raise RuntimeError("Missing JPEG for metadata rows, e.g. "+repr(raw.loc[raw.image_path.isna(),"image_id"].head().tolist()))
raw["image_path"]=raw.image_path.map(str)
print("Rows",len(raw),"lesions",raw.lesion_id.nunique(),"classes",raw.label.nunique())
if len(raw)!=10015: print("Notice: canonical HAM10000 has 10,015 rows; matched archive has",len(raw))
display(raw[["age","sex","localization","dx_type"]].describe(include="all").T)
display(raw.label.value_counts().reindex(CLASSES).rename("image_count").to_frame())
display(raw[["age","sex","localization","dx_type"]].isna().sum().rename("missing").to_frame())
fig,ax=plt.subplots(2,2,figsize=(14,10))
raw.label.value_counts().reindex(CLASSES).plot.bar(ax=ax[0,0],color="#28776c",title="Images per label")
raw.groupby("label").age.median().reindex(CLASSES).plot.bar(ax=ax[0,1],color="#d08a50",title="Median age per label")
pd.crosstab(raw.label,raw.sex,normalize="index").reindex(CLASSES).plot.bar(stacked=True,ax=ax[1,0],title="Sex share within label")
pd.crosstab(raw.label,raw.localization,normalize="index").reindex(CLASSES).plot.bar(stacked=True,ax=ax[1,1],title="Location share within label")
for a in ax.flat: a.tick_params(axis="x",rotation=35)
ax[1,0].legend(title="sex",bbox_to_anchor=(1,1)); ax[1,1].legend(title="location",bbox_to_anchor=(1,1))
plt.tight_layout(); plt.show()
for feature in ["sex","localization","age","dx_type"]:
    print("Counts by",feature); display(pd.crosstab(raw[feature].fillna("unknown"),raw.label).reindex(columns=CLASSES,fill_value=0))

## 3. Check duplicates and make grouped data splits

Exact duplicate image bytes and all images sharing a lesion ID are assigned to a connected split group. Stratified grouped partitions prevent those groups crossing train, validation and test. HAM10000 does not include patient IDs, so this is lesion/duplicate separation, not a verified patient split.

In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""): h.update(block)
    return h.hexdigest()
raw["image_sha256"]=[sha256_file(p) for p in tqdm(raw.image_path,desc="Checking exact duplicates")]
parent={}
def find(x):
    parent.setdefault(x,x)
    if parent[x]!=x: parent[x]=find(parent[x])
    return parent[x]
def union(a,b):
    a,b=find(a),find(b)
    if a!=b: parent[b]=a
for lesion in raw.lesion_id: find(lesion)
for _,g in raw.groupby("image_sha256"):
    ls=g.lesion_id.tolist()
    for x in ls[1:]: union(ls[0],x)
raw["group_id"]=raw.lesion_id.map(find)
raw["age_group"]=pd.cut(raw.age,[-1,19,39,59,79,120],labels=["0-19","20-39","40-59","60-79","80+"]).astype("object")
raw["age_group"]=raw.age_group.where(raw.age_group.notna(),"unknown")
outer=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
trv,te=next(outer.split(raw,raw.label,groups=raw.group_id)); trainval=raw.iloc[trv].reset_index(drop=True); test=raw.iloc[te].copy()
inner=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED+1)
tr,va=next(inner.split(trainval,trainval.label,groups=trainval.group_id))
train=trainval.iloc[tr].copy(); val=trainval.iloc[va].copy()
for f,n in [(train,"train"),(val,"validation"),(test,"test")]: f["split"]=n
all_df=pd.concat([train,val,test],ignore_index=True)
if all_df.groupby("group_id").split.nunique().max()!=1: raise RuntimeError("Leakage: a lesion/duplicate group crosses partitions.")
for f,n in [(train,"train"),(val,"validation"),(test,"test")]:
    if set(f.label)!=set(CLASSES): raise RuntimeError(n+" partition lacks a class; change SEED and rerun.")
display(pd.crosstab(all_df.split,all_df.label).reindex(index=["train","validation","test"],columns=CLASSES,fill_value=0))
print("Image counts:",all_df.split.value_counts().to_dict(),"lesions:",all_df.groupby("split").lesion_id.nunique().to_dict(),
      "unique exact files:",raw.image_sha256.nunique(),"duplicate rows:",int(raw.image_sha256.duplicated(keep=False).sum()))
manifest_bytes=all_df[["image_id","lesion_id","image_sha256","label","split"]].sort_values("image_id").to_csv(index=False).encode()
MANIFEST_HASH=hashlib.sha256(manifest_bytes).hexdigest(); print("Evaluation manifest SHA-256:",MANIFEST_HASH)

## 4. Train image-only and metadata-fusion models

The metadata model combines EfficientNet image features with age, sex and localization. Some training rows randomly hide metadata so the model remains usable when users leave optional fields blank. Both candidates share the same grouped split; validation balanced accuracy selects the app model. Comparison is against an image-only baseline. The test set is not used for model selection.

In [ ]:
age_train=train.age.dropna(); AGE_LO=float(age_train.quantile(.01)); AGE_HI=float(age_train.quantile(.99)); AGE_MED=float(age_train.median()); AGE_STD=float(age_train.clip(AGE_LO,AGE_HI).std())
if not np.isfinite(AGE_STD) or AGE_STD<1e-6: AGE_STD=1.
SEX_CATS=sorted(set(train.sex)-{"unknown"})+["unknown"]; LOC_CATS=sorted(set(train.localization)-{"unknown"})+["unknown"]
SCHEMA={"age_lower":AGE_LO,"age_upper":AGE_HI,"age_median":AGE_MED,"age_std":AGE_STD,
        "sex_categories":SEX_CATS,"localization_categories":LOC_CATS,
        "feature_names":["age_z","age_missing"]+["sex="+x for x in SEX_CATS]+["localization="+x for x in LOC_CATS]}
def encode_meta(age,sex,loc):
    a=pd.to_numeric(pd.Series([age]),errors="coerce").iloc[0]; miss=not np.isfinite(a)
    if miss: a=AGE_MED
    a=float(np.clip(a,AGE_LO,AGE_HI)); sex=str(sex).strip().lower() if sex is not None else "unknown"; loc=str(loc).strip().lower() if loc is not None else "unknown"
    if sex not in SEX_CATS: sex="unknown"
    if loc not in LOC_CATS: loc="unknown"
    return np.asarray([(a-AGE_MED)/AGE_STD,float(miss)]+[float(sex==x) for x in SEX_CATS]+[float(loc==x) for x in LOC_CATS],dtype=np.float32)
for frame in [train,val,test]: frame["meta_vec"]=[encode_meta(r.age,r.sex,r.localization) for r in frame.itertuples()]
MDIM=len(SCHEMA["feature_names"]); mean,std=(.485,.456,.406),(.229,.224,.225)
train_tf=transforms.Compose([transforms.Resize(256),transforms.RandomResizedCrop(224,scale=(.82,1.)),
    transforms.RandomHorizontalFlip(),transforms.RandomVerticalFlip(p=.25),transforms.ColorJitter(.08,.08,.05,.015),
    transforms.ToTensor(),transforms.Normalize(mean,std)])
eval_tf=transforms.Compose([transforms.Resize(224),transforms.CenterCrop(224),transforms.ToTensor(),transforms.Normalize(mean,std)])
class LesionDataset(Dataset):
    def __init__(self,frame,tf,meta=True,drop=.0): self.f=frame.reset_index(drop=True); self.tf=tf; self.meta=meta; self.drop=drop
    def __len__(self): return len(self.f)
    def __getitem__(self,i):
        r=self.f.iloc[i]
        with Image.open(r.image_path) as im: x=self.tf(im.convert("RGB"))
        m=torch.tensor(r.meta_vec,dtype=torch.float32)
        if not self.meta: m=torch.zeros_like(m)
        elif self.drop and random.random()<self.drop: m=torch.tensor(encode_meta(None,None,None))
        return x,torch.tensor(C2I[r.label]),m
def loader(frame,tf,meta,shuffle=False,drop=0):
    return DataLoader(LesionDataset(frame,tf,meta,drop),batch_size=BS,shuffle=shuffle,num_workers=WORKERS,
                      pin_memory=DEVICE.type=="cuda",persistent_workers=WORKERS>0)
class SkinNet(nn.Module):
    def __init__(self,mdim=0,pretrained=True):
        super().__init__(); self.mdim=mdim
        self.encoder=timm.create_model("efficientnet_b0",pretrained=pretrained,num_classes=0)
        d=self.encoder.num_features
        self.fusion=nn.Sequential(nn.Linear(d+mdim,512),nn.LayerNorm(512),nn.ReLU(),nn.Dropout(.35),nn.Linear(512,7)) if mdim else nn.Linear(d,7)
    def forward(self,x,m=None):
        f=self.encoder(x)
        if self.mdim: f=torch.cat([f,m if m is not None else torch.zeros((len(x),self.mdim),device=x.device)],dim=1)
        return self.fusion(f)
def train_candidate(use_meta,name):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    loaders={"train":loader(train,train_tf,use_meta,True,.25 if use_meta else 0),
             "val":loader(val,eval_tf,use_meta)}
    counts=train.label.value_counts().reindex(CLASSES).to_numpy()
    weights=torch.tensor(len(train)/(7*counts),dtype=torch.float32,device=DEVICE)
    model=SkinNet(MDIM if use_meta else 0).to(DEVICE); loss_fn=nn.CrossEntropyLoss(weight=weights)
    opt=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=1e-4)
    sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",factor=.5,patience=2,min_lr=1e-6)
    scaler=torch.cuda.amp.GradScaler(enabled=DEVICE.type=="cuda")
    best=-1.; epoch_best=0; stale=0; hist=[]; path=OUT/(name+"_best.pt")
    for epoch in range(1,EPOCHS+1):
        model.train(); loss_total=0.
        for x,y,m in tqdm(loaders["train"],desc=name+" epoch "+str(epoch),leave=False):
            x,y,m=x.to(DEVICE),y.to(DEVICE),m.to(DEVICE); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda",enabled=DEVICE.type=="cuda"): logits=model(x,m if use_meta else None); loss=loss_fn(logits,y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); loss_total+=float(loss.detach())*len(y)
        model.eval(); truth=[]; pred=[]
        with torch.no_grad():
            for x,y,m in loaders["val"]:
                z=model(x.to(DEVICE),m.to(DEVICE) if use_meta else None)
                truth.extend(y.tolist()); pred.extend(z.argmax(1).cpu().tolist())
        score=float(balanced_accuracy_score(truth,pred)); row={"epoch":epoch,"train_loss":loss_total/len(train),"val_balanced_accuracy":score,
              "val_macro_f1":float(f1_score(truth,pred,average="macro",zero_division=0))}; hist.append(row); sched.step(score); print(name,row)
        if score>best+1e-5:
            best=score; epoch_best=epoch; stale=0; torch.save(model.state_dict(),path)
        else:
            stale+=1
            if stale>=PATIENCE: print("Early stop",name,"epoch",epoch); break
    model.load_state_dict(torch.load(path,map_location=DEVICE,weights_only=True))
    return model,{"best_epoch":epoch_best,"best_validation_balanced_accuracy":best,"history":hist}
image_model,image_info=train_candidate(False,"image_only")
fusion_model,fusion_info=train_candidate(True,"metadata_fusion")
chosen_name="metadata_fusion" if fusion_info["best_validation_balanced_accuracy"]>=image_info["best_validation_balanced_accuracy"] else "image_only"
chosen=fusion_model if chosen_name=="metadata_fusion" else image_model; USE_META=chosen_name=="metadata_fusion"
print("Validation-selected:",chosen_name)
json.dump({"image_only":image_info,"metadata_fusion":fusion_info},open(OUT/"training_history.json","w"),indent=2)

## 5. Calibrate on validation; evaluate the locked test set

Temperature scaling is fitted only on validation logits. Then the selected model is evaluated on the test partition, including per-class metrics, confusion matrices, Brier score, log loss, and top-label ECE. A second test pass with metadata set to unknown is a prespecified missing-feature ablation on the same images, not an independent test.

In [ ]:
def collect(model,frame,use_meta):
    ld=loader(frame,eval_tf,use_meta); model.eval(); zs=[]; ys=[]
    with torch.no_grad():
        for x,y,m in ld: zs.append(model(x.to(DEVICE),m.to(DEVICE) if use_meta else None).float().cpu().numpy()); ys.extend(y.numpy())
    return np.concatenate(zs),np.asarray(ys)
vz,vy=collect(chosen,val,USE_META)
lt=torch.zeros(1,device=DEVICE,requires_grad=True); zt=torch.tensor(vz,dtype=torch.float32,device=DEVICE); yt=torch.tensor(vy,dtype=torch.long,device=DEVICE)
ot=torch.optim.LBFGS([lt],lr=.1,max_iter=50)
def closure():
    ot.zero_grad(); loss=F.cross_entropy(zt/torch.exp(lt).clamp(.05,20),yt); loss.backward(); return loss
ot.step(closure); TEMP=float(torch.exp(lt).clamp(.05,20).detach().cpu()); print("Validation temperature:",TEMP)
def probs(z,t=1):
    z=z/float(t); z=z-z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)
def ece(y,p,bins=15):
    c=p.max(1); pred=p.argmax(1); result=0.
    for lo,hi in zip(np.linspace(0,1,bins+1)[:-1],np.linspace(0,1,bins+1)[1:]):
        m=(c>=lo)&(c<(hi if hi<1 else hi+1e-9))
        if m.any(): result+=m.mean()*abs((pred[m]==y[m]).mean()-c[m].mean())
    return float(result)
def pack(y,p):
    pred=p.argmax(1)
    return {"images":int(len(y)),"accuracy":float(accuracy_score(y,pred)),
      "balanced_accuracy":float(balanced_accuracy_score(y,pred)),"macro_f1":float(f1_score(y,pred,average="macro",zero_division=0)),
      "macro_ovr_auc":float(roc_auc_score(y,p,labels=np.arange(7),multi_class="ovr",average="macro")),
      "log_loss":float(log_loss(y,p,labels=np.arange(7))),"multiclass_brier":float(np.mean(np.sum((p-np.eye(7)[y])**2,axis=1))),
      "top_label_ece_15_bins":ece(y,p)}
tz,ty=collect(chosen,test,USE_META); tp=probs(tz,TEMP); metrics=pack(ty,tp)
cmc=confusion_matrix(ty,tp.argmax(1),labels=np.arange(7)); cmn=confusion_matrix(ty,tp.argmax(1),labels=np.arange(7),normalize="true")
pc=classification_report(ty,tp.argmax(1),labels=np.arange(7),target_names=CLASSES,output_dict=True,zero_division=0)
metrics.update({"classes":CLASSES,"per_class":pc,"test_class_support":{c:int((ty==i).sum()) for i,c in enumerate(CLASSES)},
 "confusion_matrix_counts":cmc.tolist(),"confusion_matrix_normalized":cmn.tolist(),"calibration":"Temperature fitted on validation only; internal diagnostic.",
 "temperature":TEMP,"selected_model":chosen_name,"evaluation_date":pd.Timestamp.utcnow().date().isoformat(),
 "evaluation_manifest_sha256":MANIFEST_HASH,"validation_scores":{"image_only":image_info["best_validation_balanced_accuracy"],"metadata_fusion":fusion_info["best_validation_balanced_accuracy"]}})
json.dump(metrics,open(OUT/"evaluation_metrics.json","w"),indent=2)
print(json.dumps({k:v for k,v in metrics.items() if k not in ["classes","per_class","test_class_support","confusion_matrix_counts","confusion_matrix_normalized"]},indent=2))
display(pd.DataFrame(pc).T)
if USE_META:
    unknown=np.stack([encode_meta(None,None,None) for _ in range(len(test))]); no_meta=[]; i=0
    chosen.eval()
    with torch.no_grad():
        for x,y,m in loader(test,eval_tf,True):
            n=len(y); mm=torch.tensor(unknown[i:i+n],device=DEVICE); no_meta.append(chosen(x.to(DEVICE),mm).float().cpu().numpy()); i+=n
    ablation=pack(ty,probs(np.concatenate(no_meta),TEMP))
else: ablation={"note":"Validation-selected model is image-only; metadata does not affect its scores."}
json.dump({"all_features":metrics,"all_metadata_unknown_same_test":ablation},open(OUT/"metadata_ablation.json","w"),indent=2)
print("Metadata ablation:",ablation)

## 6. Subgroups and visual diagnostics

Subgroup metrics include support and omit groups under 10 images from the compact table. These are descriptive dataset results, not fairness claims or estimates for other populations. dx_type remains analysis-only.

In [ ]:
def subgroup(frame,y,p,col):
    d=frame.reset_index(drop=True).copy(); d["truth"]=y; d["pred"]=p.argmax(1); rows=[]
    for key,g in d.groupby(col,dropna=False):
        if len(g)>=10: rows.append({"group":str(key),"support":len(g),"accuracy":accuracy_score(g.truth,g.pred),
          "balanced_accuracy":balanced_accuracy_score(g.truth,g.pred),"macro_f1":f1_score(g.truth,g.pred,average="macro",zero_division=0)})
    return pd.DataFrame(rows).sort_values("support",ascending=False)
subgroups={}
for col in ["sex","age_group","localization","dx_type"]:
    tb=subgroup(test,ty,tp,col); subgroups[col]=tb.to_dict(orient="records"); print("Test groups:",col); display(tb)
json.dump(subgroups,open(OUT/"subgroup_metrics.json","w"),indent=2)
fig,ax=plt.subplots(2,2,figsize=(14,10))
im=ax[0,0].imshow(cmn,vmin=0,vmax=1,cmap="Blues"); ax[0,0].set(xticks=range(7),yticks=range(7),xticklabels=CLASSES,yticklabels=CLASSES,xlabel="Predicted",ylabel="True",title="Row-normalized test confusion")
for i in range(7):
    for j in range(7): ax[0,0].text(j,i,str(cmc[i,j]),ha="center",va="center",fontsize=8)
fig.colorbar(im,ax=ax[0,0])
conf=tp.max(1); corr=(tp.argmax(1)==ty); xs=[];ys=[]
for lo,hi in zip(np.linspace(0,1,11)[:-1],np.linspace(0,1,11)[1:]):
    m=(conf>=lo)&(conf<(hi if hi<1 else hi+1e-9))
    if m.any(): xs.append(conf[m].mean()); ys.append(corr[m].mean())
ax[0,1].plot([0,1],[0,1],"--",color="gray"); ax[0,1].plot(xs,ys,"o-"); ax[0,1].set(xlim=(0,1),ylim=(0,1),xlabel="Mean top score",ylabel="Accuracy",title="Test reliability by score bin")
for name,info in [("image-only",image_info),("metadata-fusion",fusion_info)]:
    h=info["history"]; ax[1,0].plot([r["epoch"] for r in h],[r["val_balanced_accuracy"] for r in h],label=name)
ax[1,0].set(xlabel="Epoch",ylabel="Validation balanced accuracy",title="Model selection"); ax[1,0].legend()
pd.Series(metrics["test_class_support"]).reindex(CLASSES).plot.bar(ax=ax[1,1],color="#d08a50",title="Test support")
plt.tight_layout(); plt.savefig(OUT/"evaluation_report.png",dpi=180); plt.show()

## 7. Export one app-ready checkpoint and result bundle

The checkpoint uses only tensors and primitive Python values, so PyTorch restricted weights-only loading works. It carries a model type, class order, feature schema and validation temperature. The app can fall back to image-only scoring when the validation-selected model is image-only. Downloading the result bundle saves to your computer and does not connect Google Drive.

In [ ]:
state={k:v.detach().cpu() for k,v in chosen.state_dict().items()}
checkpoint={"model_type":"efficientnet_b0_metadata_fusion_v1" if USE_META else "efficientnet_b0_image_only_v1",
 "model_state":state,"model_name":"efficientnet_b0","classes":CLASSES,"image_size":IMG_SIZE,
 "metadata_schema":SCHEMA if USE_META else None,"temperature":TEMP,"selected_by":"validation_balanced_accuracy"}
torch.save(checkpoint,OUT/"best.pt")
torch.save({k:v.detach().cpu() for k,v in image_model.state_dict().items()},OUT/"image_only.pt")
torch.save({k:v.detach().cpu() for k,v in fusion_model.state_dict().items()},OUT/"metadata_fusion.pt")
sha=hashlib.sha256((OUT/"best.pt").read_bytes()).hexdigest()
model_meta={"model":checkpoint["model_name"],"model_type":checkpoint["model_type"],"library":"timm","library_version":timm.__version__,
 "torch_version":torch.__version__,"classes":CLASSES,"image_size":IMG_SIZE,"normalization":{"mean":list(mean),"std":list(std)},
 "metadata_schema":checkpoint["metadata_schema"],"selected_model":chosen_name,"validation_scores":metrics["validation_scores"],
 "test_metrics_file":"evaluation_metrics.json","seed":SEED,"artifact_sha256":sha,"evaluation_manifest_sha256":MANIFEST_HASH,
 "evaluation_date":metrics["evaluation_date"],"dataset_manifest_sha256":hashlib.sha256(META_PATH.read_bytes()).hexdigest(),
 "dataset":"HAM10000","dataset_doi":"10.7910/DVN/DBW86T",
 "dataset_citation":"Tschandl et al. (2018), The HAM10000 dataset, Scientific Data 5, 180161. doi:10.1038/sdata.2018.161",
 "dataset_license_note":"Non-commercial use only; see Harvard Dataverse terms.",
 "feature_engineering":{"used":["age","sex","localization"] if USE_META else [],
 "analysis_only":["dx_type","image_id","lesion_id","image_sha256","group_id"],
 "age":"Train-only clipping, median imputation, standardization and missing flag; unknown supported.",
 "categoricals":"Train vocabulary one-hot encoding with unknown category; metadata dropout 0.25 during fusion training."},
 "split_group":"lesion_id plus exact image SHA-256 connected groups; no patient ID available",
 "split_counts":{k:int(v) for k,v in all_df.split.value_counts().items()},
 "intended_use":"Research only; not validated for diagnosis or triage."}
json.dump(model_meta,open(OUT/"model_metadata.json","w"),indent=2); json.dump(SCHEMA,open(OUT/"metadata_schema.json","w"),indent=2)
all_df[["image_id","lesion_id","image_sha256","label","split"]].sort_values("image_id").to_csv(OUT/"evaluation_manifest.csv",index=False)
bundle=Path("/content/skinlesion_results.zip")
names=["best.pt","image_only.pt","metadata_fusion.pt","training_history.json","evaluation_metrics.json","model_metadata.json","metadata_schema.json","metadata_ablation.json","subgroup_metrics.json","evaluation_manifest.csv","evaluation_report.png"]
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for name in names:
        p=OUT/name
        if p.exists(): z.write(p,arcname=name)
print("Bundle ready:",bundle,"size",round(bundle.stat().st_size/1e6,1),"MB")
print("Selected",chosen_name,"| test balanced accuracy",round(metrics["balanced_accuracy"],4),"| checkpoint SHA-256",sha)
print("Save evaluation_manifest.csv with this exact test result.")
from google.colab import files
files.download(str(bundle))